# Large Dataset Cleaning

Downloads and preprocesses the  dataset for Axis 2 priority classification experiments.


In [1]:
import re
from pathlib import Path
from datasets import load_dataset

In [2]:
PROJECT_ROOT = Path.cwd().parents[2]

RAW_PATH       = PROJECT_ROOT / "data" / "raw"       / "large_customer_support_tickets.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "large_dataset_preprocessed.csv"

## 1. Load & Save Raw

In [3]:
ds  = load_dataset("Tobi-Bueck/customer-support-tickets", split="train")
raw = ds.to_pandas()

print("Raw shape:", raw.shape)
raw[["subject","body","priority","type","language"]].head()

raw.to_csv(RAW_PATH, index=False, encoding="utf-8")

Raw shape: (61765, 16)


## 2. Filter & Select

In [4]:
df = raw[raw["language"] == "en"][["subject","body","priority","type","tag_1","tag_2"]].copy().reset_index(drop=True)
df.shape

(28261, 6)

## 3. Text Cleaning

Lighter cleaning than the synthetic dataset: no template removal needed.
Focus on normalising whitespace and removing noise characters.

In [5]:
before = len(df)
df = df.dropna(subset=["body", "priority"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with missing body/priority. Remaining: {len(df)}")

Dropped 1 rows with missing body/priority. Remaining: 28260


In [6]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)            # remove HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text) # remove URLs
    text = re.sub(r"\S+@\S+\.\S+", " ", text)    # remove emails
    text = re.sub(r"[^\w\s.,!?'-]", " ", text)   # keep basic punctuation
    text = re.sub(r"\s+", " ", text).strip()         # normalise whitespace
    return text

df["subject_clean"] = df["subject"].fillna("").apply(clean_text)
df["body_clean"] = df["body"].apply(clean_text)
df["text_cleaned"] = (df["subject_clean"] + " " + df["body_clean"]).str.strip()

In [7]:
df["text_length"] = df["text_cleaned"].apply(lambda x: len(x.split()))
print(df["text_length"].describe())
before = len(df)
df = df[df["text_length"] >= 5].reset_index(drop=True)

count    28260.000000
mean        60.598726
std         31.729027
min          1.000000
25%         35.000000
50%         59.000000
75%         84.000000
max        299.000000
Name: text_length, dtype: float64


## 4. Save Processed

In [8]:
out = df[["subject","body","text_cleaned","priority","type","tag_1","tag_2","text_length"]]
out.to_csv(PROCESSED_PATH, index=False, encoding="utf-8")
